In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install duckdb -q

In [4]:
import duckdb
import pandas as pd

parquet_path = "/content/drive/MyDrive/GENIUS/IndividualsAndHouseholdsProgramValidRegistrationsV2.parquet"

con = duckdb.connect()

In [5]:
con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{parquet_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,incidentTypeCode,VARCHAR,YES,None,None,None
1,declarationDate,DATE,YES,None,None,None
2,disasterNumber,SMALLINT,YES,None,None,None
3,county,VARCHAR,YES,None,None,None
4,fips,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
95,verifiedOwnership,BOOLEAN,YES,None,None,None
96,verifiedOccupancy,BOOLEAN,YES,None,None,None
97,appliedDate,DATE,YES,None,None,None
98,lastRefresh,TIMESTAMP WITH TIME ZONE,YES,None,None,None


In [7]:
query = f"""
copy (
  select
    damagedStateAbbreviation as damaged_state_abbreviation,
    county,
    fips,
    damagedZipCode as damaged_zip_code,
    incidentTypeCode as incident_type_code,

    ownRent as own_rent,
    grossIncome as gross_income,
    householdComposition as household_composition,

    homeDamage as home_damage,
    floodDamage as flood_damage,
    autoDamage as auto_damage,
    emergencyNeeds as emergency_needs,
    foodNeed as food_need,
    shelterNeed as shelter_need,

    count(*) as total_cases,

    sum(case when ihpEligible = true then 1 else 0 end) as eligible_cases,

    sum(case when ihpEligible = true then 1 else 0 end) * 1.0 / count(*) as eligibility_rate,

    avg(ihpAmount) as avg_ihp_amount,
    median(ihpAmount) as median_ihp_amount,
    quantile_cont(ihpAmount, 0.25) as p25_ihp_amount,
    quantile_cont(ihpAmount, 0.75) as p75_ihp_amount,

    avg(haAmount) as avg_ha_amount,
    avg(onaAmount) as avg_ona_amount,
    avg(rentalAssistanceAmount) as avg_rental_assistance_amount,
    avg(repairAmount) as avg_repair_amount,
    avg(personalPropertyAmount) as avg_personal_property_amount,

    avg(rpfvl) as avg_rpfvl,
    avg(ppfvl) as avg_ppfvl

  from read_parquet('{parquet_path}')

  where damagedZipCode is not null
    and damagedStateAbbreviation is not null
    and county is not null
    and ihpAmount is not null

  group by
    damagedStateAbbreviation,
    county,
    fips,
    damagedZipCode,
    incidentTypeCode,
    ownRent,
    grossIncome,
    householdComposition,
    homeDamage,
    floodDamage,
    autoDamage,
    emergencyNeeds,
    foodNeed,
    shelterNeed

  having count(*) >= 10
) to '/content/drive/MyDrive/GENIUS/fema_ihp_assistance_summary.csv' with (header, delimiter ',');
"""

con.execute(query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))